In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
import time
from anthropic import RateLimitError
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None, max_retries=3):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    for attempt in range(max_retries):
        try:
            return client.messages.create(**params)
        except RateLimitError:
            wait = 2 ** attempt * 10  # 10s, 20s, 40s
            print(f"Rate limited. Waiting {wait}s before retry {attempt + 1}/{max_retries}...")
            time.sleep(wait)

    raise RuntimeError("Exceeded max retries due to rate limiting")


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
from text_editor import get_text_edit_schema, run_tool, run_tools, text_editor_tool

web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    #"allowed_domains": ["nih.gov"]
}

text_editor_schema = get_text_edit_schema()

bash_tool_schema = {"type": "bash_20250124", "name": "bash"}

In [6]:
from anthropic.types import TextBlock

tools = [web_search_schema, text_editor_schema, bash_tool_schema]

messages = []
add_user_message(
    messages,
    """
Can you udpate the readme for this repo in Git?
    """,
)

MAX_ITERATIONS = 10

for i in range(MAX_ITERATIONS):
    print(f"\n--- Turn {i + 1} ---")
    response = chat(messages, tools=tools)
    add_assistant_message(messages, response)

    for block in response.content:
        if isinstance(block, TextBlock):
            print(block.text)

    if response.stop_reason != "tool_use":
        break

    tool_results = run_tools(response)
    add_user_message(messages, tool_results)
else:
    print("Reached max iterations — stopping.")


--- Turn 1 ---
I'll help you update the README for this repository. Let me first check what's currently in the repository and locate the README file.
[tool_use] name='bash' input={'command': 'pwd'}
[tool_result] 'c:\\Users\\jayak\\OneDrive\\Documents\\Projects\\Claude Training Client\\src'
[tool_use] name='bash' input={'command': 'ls -la'}
[tool_result] 'Volume in drive C is Windows\n Volume Serial Number is 2CCE-8BE6\n\n Directory of c:\\Users\\jayak\\OneDrive\\Documents\\Projects\\Claude Training Client\\src\nSTDERR: File Not Found'

--- Turn 2 ---
[tool_use] name='bash' input={'command': 'cd .. && pwd && ls -la'}
[tool_result] "Command rejected: Shell operator '&&' is not allowed"

--- Turn 3 ---
[tool_use] name='bash' input={'command': 'cd ..'}
[tool_result] ''
[tool_use] name='bash' input={'command': 'ls -la'}
[tool_result] 'Volume in drive C is Windows\n Volume Serial Number is 2CCE-8BE6\n\n Directory of c:\\Users\\jayak\\OneDrive\\Documents\\Projects\\Claude Training Client\\sr